<a href="https://colab.research.google.com/github/AlvinR21/Northstar-analysis/blob/main/SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
install.packages("sqldf")
install.packages("dplyr")

library(sqldf)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [21]:
customers <- read.csv("customers_clean.csv")
drivers <- read.csv("drivers_clean.csv")
vehicles <- read.csv("vehicles_clean.csv")
hubs <- read.csv("hubs_clean.csv")
orders <- read.csv("orders_clean.csv")
deliveries <- read.csv("deliveries_clean.csv")
incidents <- read.csv("incidents_clean.csv")
complaints <- read.csv("complaints_clean.csv")
app_events <- read.csv("app_events_clean.csv")

In [22]:
#Show how many rows each table has
print(paste("customers rows:", nrow(customers)))
print(paste("orders rows:", nrow(orders)))
print(paste("deliveries rows:", nrow(deliveries)))

[1] "customers rows: 650"
[1] "orders rows: 1250"
[1] "deliveries rows: 950"


In [23]:
#Show the first 5 rows of the deliveries table
sqldf("SELECT * FROM deliveries LIMIT 5")

delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost,delivery_duration_hours,promised_window_hours,is_late,dispatch_day_of_week,hub_zone,hub_type
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>
DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05,22.149973,6,True,Tuesday,Central,Control
DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41,-1.100000,2,False,Saturday,South,Dispatch
DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51,1.108991,2,False,Monday,South,Dispatch
DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62,23.985584,24,False,Friday,South,Dispatch
DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22,4.042814,6,False,Sunday,North,Dispatch


In [24]:
#Show only the deliveries that were late, sorted by duration (longest first)
sqldf("
  SELECT delivery_id, hub_id, delivery_duration_hours, promised_window_hours
  FROM deliveries
  WHERE is_late = 'True'
  ORDER BY delivery_duration_hours DESC
  LIMIT 10
")

delivery_id,hub_id,delivery_duration_hours,promised_window_hours
<chr>,<chr>,<dbl>,<int>
DL00386,H05,43.45692,24
DL00387,H01,42.12664,24
DL00033,H06,40.84033,24
DL00530,H01,39.56579,24
DL00026,H04,38.54606,24
DL00806,H07,37.64825,24
DL00497,H08,37.29338,24
DL00472,H01,37.10086,24
DL00775,H05,36.73224,24


H01 has the most extreme late deliveries (the top 10 worst single deliveries), and H06 (from Python Cell 42) has the highest average late rate. These are different operational issues that need different solutions.

In [25]:
#For each hub, count total deliveries and late deliveries, and calculate the late rate
sqldf("
  SELECT hub_id,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN is_late = 'True' THEN 1 ELSE 0 END) AS late_deliveries,
         ROUND(
           1.0 * SUM(CASE WHEN is_late = 'True' THEN 1 ELSE 0 END) / COUNT(*),
           3
         ) AS late_rate
  FROM deliveries
  GROUP BY hub_id
  ORDER BY late_rate DESC
")

hub_id,total_deliveries,late_deliveries,late_rate
<chr>,<int>,<int>,<dbl>
H06,104,59,0.567
H05,115,59,0.513
H04,127,60,0.472
H08,128,60,0.469
H01,136,58,0.426
H02,106,44,0.415
H03,119,49,0.412
H07,115,46,0.400


H06 has 56.7% late delivery rate — highest of all 8 hubs. Confirmed via SQL GROUP BY query, matching Python result.

In [26]:
#From the previouse query, but joined with hubs to show name, zone, and type
sqldf("
  SELECT h.hub_id,
         h.hub_name,
         h.zone,
         h.hub_type,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN d.is_late = 'True' THEN 1 ELSE 0 END) AS late_deliveries,
         ROUND(
           1.0 * SUM(CASE WHEN d.is_late = 'True' THEN 1 ELSE 0 END) / COUNT(*),
           3
         ) AS late_rate
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_id, h.hub_name, h.zone, h.hub_type
  ORDER BY late_rate DESC
")

hub_id,hub_name,zone,hub_type,total_deliveries,late_deliveries,late_rate
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>
H06,Airport Hub,Airport,Dispatch,104,59,0.567
H05,Central Core,Central,Control,115,59,0.513
H04,West Gate,West,Dispatch,127,60,0.472
H08,Midtown Relay,Central,Charging,128,60,0.469
H01,North Exchange,North,Dispatch,136,58,0.426
H02,South Link,South,Dispatch,106,44,0.415
H03,East Dock,East,Warehouse,119,49,0.412
H07,Riverside Hub,Riverside,Warehouse,115,46,0.400


Hub H06 (Airport Hub), a Dispatch-type hub serving the Airport zone, has the highest consistent late-delivery rate at 56.7%. As a Dispatch hub, its main job is to assign routes and send out vehicles. This performance suggests that there are problems with scheduling or workload management, not with the availability of vehicles or drivers. This evidence directly backs up the operations director's idea that some hubs don't work as well as they should.

In [27]:
#For each driver, count their deliveries, average overrides, late rate, and avg customer rating
sqldf("
  SELECT d.driver_id,
         dr.base_zone,
         dr.years_experience,
         COUNT(*) AS total_deliveries,
         ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides,
         ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating,
         ROUND(
           1.0 * SUM(CASE WHEN d.is_late = 'True' THEN 1 ELSE 0 END) / COUNT(*),
           3
         ) AS late_rate
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id, dr.base_zone, dr.years_experience
  HAVING COUNT(*) >= 3
  ORDER BY avg_overrides DESC
  LIMIT 10
")

driver_id,base_zone,years_experience,total_deliveries,avg_overrides,avg_rating,late_rate
<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>
D127,Central,10,6,2.83,4.10,0.167
D062,South,10,3,2.00,3.82,1.000
D069,North,2,7,2.00,3.94,0.429
D085,North,9,4,2.00,3.42,0.500
D105,Riverside,2,7,2.00,4.21,0.286
D124,North,4,4,2.00,3.41,0.500
D130,West,8,8,2.00,3.80,0.375
D139,South,10,5,2.00,4.08,0.400
D028,North,11,7,1.86,3.54,0.571


The top driver goes over 2.38 routes per delivery on average, but their customer rating is 4.10 and their late rate is only 16.7%. This goes against the case study's idea that overrides are a sign of poor performance, at least for the top driver. Some manual overrides may show real route knowledge instead of trying to avoid it.


In [28]:
#To count complaints per zone, broken down by complaint type
sqldf("
  SELECT c.home_zone,
         cp.complaint_type,
         COUNT(*) AS complaint_count,
         ROUND(AVG(cp.resolution_days), 1) AS avg_resolution_days,
         ROUND(AVG(cp.compensation_amount), 2) AS avg_compensation
  FROM complaints cp
  JOIN customers c ON cp.customer_id = c.customer_id
  GROUP BY c.home_zone, cp.complaint_type
  ORDER BY complaint_count DESC
  LIMIT 15
")

home_zone,complaint_type,complaint_count,avg_resolution_days,avg_compensation
<chr>,<chr>,<int>,<dbl>,<dbl>
North,Delay,18,8.0,21.03
North,AppIssue,17,7.5,22.26
South,Delay,16,6.3,10.74
East,Delay,15,6.6,17.92
North,MissedPickup,15,7.1,15.12
South,MissedPickup,15,6.2,22.31
Riverside,Delay,14,8.0,20.21
Airport,Delay,13,7.3,17.07
Central,Delay,13,7.8,15.98


Customers in the North zone file the most complaints (18 Delay-type), but H06 (the hub with the worst performance by late rate) is in the Airport zone. Operational failure and customer complaints does not happen in the same places. This evidence backs up the customer experience director's worry about "not properly connecting complaints, missed journeys, failed deliveries, and driver incidents into one view."

In [29]:
#For each incident type, show count, severity breakdown, and how long it takes to resolve
sqldf("
  SELECT incident_type,
         COUNT(*) AS incident_count,
         ROUND(AVG(resolved_hours), 1) AS avg_resolved_hours,
         SUM(CASE WHEN severity = 'High' THEN 1 ELSE 0 END) AS high_severity,
         SUM(CASE WHEN severity = 'Critical' THEN 1 ELSE 0 END) AS critical_severity,
         SUM(CASE WHEN resolution_status = 'Open' THEN 1 ELSE 0 END) AS still_open
  FROM incidents
  GROUP BY incident_type
  ORDER BY avg_resolved_hours DESC
")

incident_type,incident_count,avg_resolved_hours,high_severity,critical_severity,still_open
<chr>,<int>,<dbl>,<int>,<int>,<int>
CustomerNoShow,44,13.8,11,5,12
RouteDeviation,43,13.6,12,4,15
TemperatureIssue,29,12.9,5,5,3
AppSyncError,31,12.5,6,1,7
BatteryAlert,36,11.7,8,2,11
ProofMissing,46,10.9,12,6,12
SafetyNearMiss,14,9.8,4,1,3
VehicleFault,37,9.2,10,3,14


There are still 12 CustomerNoShow cases open, and it takes an average of 13.8 hours to resolve them. It should only take a few minutes to solve a "no-show" (the driver confirms and the case is closed). So, 13.8 hours means that the process flow between dispatch, drivers, and customer support systems is broken. Direct proof that NorthStar's operational data flow is broken.


In [30]:
#Find drivers whose late rate is worse than overall average late rate
sqldf("
  SELECT driver_id,
         total_deliveries,
         late_rate
  FROM (
    SELECT d.driver_id,
           COUNT(*) AS total_deliveries,
           ROUND(
             1.0 * SUM(CASE WHEN d.is_late = 'True' THEN 1 ELSE 0 END) / COUNT(*),
             3
           ) AS late_rate
    FROM deliveries d
    GROUP BY d.driver_id
    HAVING COUNT(*) >= 5
  ) AS driver_stats
  WHERE late_rate > 0.458
  ORDER BY late_rate DESC
  LIMIT 15
")

driver_id,total_deliveries,late_rate
<chr>,<int>,<dbl>
D100,8,0.875
D023,6,0.833
D094,5,0.800
D144,5,0.800
D162,5,0.800
D104,7,0.714
D011,6,0.667
D082,6,0.667
D091,6,0.667


Driver D100 has late 87.5% of the time, which is almost two times the average. At least 15 drivers do worse than the average for their job. Along with Cell 27's finding that route-overriding does not predict poor performance, this suggests that NorthStar needs to manage driver performance on an individual basis rather than having a blanket policy on overrides.